# Cats and Dogs Classification

A course project comparing dense and convolutional neural networks, with a TensorFlow.js webcam demo. See the README for setup, model details and the results from the original training run.

## 1. Environment and dataset

In [ ]:
!pip install tensorflowjs

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds


setattr(tfds.image_classification.cats_vs_dogs, '_URL',"https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
# Load the cats_vs_dogs dataset from TensorFlow Datasets
data, metadata = tfds.load('cats_vs_dogs', as_supervised=True, with_info=True)

In [ ]:
# Display the data to check it
metadata

In [ ]:
# Display examples from the set
tfds.show_examples(data['train'], metadata)

## 2. Image preprocessing

Resize to 100 × 100, convert to grayscale and normalize pixel values.

In [ ]:
# Manipulate and visualize the image dataset:
# Resize to SIZE_IMG (100x100) and convert to black and white (grayscale) for visualization only.

import matplotlib.pyplot as plt
import cv2

plt.figure(figsize=(20,20))

SIZE_IMG=100

for i, (image, label) in enumerate(data['train'].take(25)):
  image = cv2.resize(image.numpy(), (SIZE_IMG, SIZE_IMG))
  image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  plt.subplot(5, 5, i+1)
  plt.xticks([])
  plt.yticks([])
  plt.imshow(image, cmap='gray')

In [ ]:
# Variable that will contain all the data pairs (image and label) already modified (black and white, 100x100)
data_train = []

In [ ]:
for i, (image, label) in enumerate(data['train']): #All the data
  image = cv2.resize(image.numpy(), (SIZE_IMG, SIZE_IMG))
  image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  image = image.reshape(SIZE_IMG, SIZE_IMG, 1) #Change size to 100,100,1
  data_train.append([image, label])

In [ ]:
#Show the data of the first index
data_train[0]

In [ ]:
#See how much data I have in the variable
len(data_train)

In [ ]:
#Prepare X (inputs) and y (labels) variables separately

X = [] #input images (pixels)
y = [] #labels (dogs or cats)

for image, label in data_train:
  X.append(image)
  y.append(label)

In [ ]:
X

In [ ]:
#Normalize the X data (images). They are converted to a floating number and divided by 255 to get 0-1 instead of 0-255.
import numpy as np

X = np.array(X).astype(float) / 255

In [ ]:
y

In [ ]:
#Convert label in simple array
y = np.array(y)

In [ ]:
X.shape

## 3. Baseline architectures

Compare a dense network, a CNN and a CNN with dropout.

In [ ]:
# Create the initial models
# Use sigmoid as the output (instead of softmax) to demonstrate how it might work with this activation function.
# Sigmoid always returns values between 0 and 1. We train the model so that, in the end, if the output is close to 0,
# it is considered a cat, and if it is close to 1, it is considered a dog.


denseLayer = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(100, 100, 1)),
  tf.keras.layers.Dense(150, activation='relu'),
  tf.keras.layers.Dense(150, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

CNNmodel = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(100, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

CNN2model = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(250, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
#Compile models. Use binary crossentropy since we have only 2 options (dog or cat)
denseLayer.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

CNNmodel.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

CNN2model.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import TensorBoard

## 4. Training without augmentation

In [ ]:
# The TensorBoard variable is passed in the "callbacks" array (other types of callbacks are also supported).
# In this case, it saves data in the specified folder at each epoch, so that later
# TensorBoard can read it to generate graphs.

tensorboardDense = TensorBoard(log_dir='logs/dense')
denseLayer.fit(X, y, batch_size=32,
                validation_split=0.15,
                epochs=100,
                callbacks=[tensorboardDense])

In [ ]:
tensorboardCNN = TensorBoard(log_dir='logs/cnn')
CNNmodel.fit(X, y, batch_size=32,
                validation_split=0.15,
                epochs=100,
                callbacks=[tensorboardCNN])

In [ ]:
tensorboardCNN2 = TensorBoard(log_dir='logs/cnn2')
CNN2model.fit(X, y, batch_size=32,
                validation_split=0.15,
                epochs=100,
                callbacks=[tensorboardCNN2])

## 5. Data augmentation

In [ ]:
# See the images of variable X without modifications due to data increase
plt.figure(figsize=(20, 8))
for i in range(10):
  plt.subplot(2, 5, i+1)
  plt.xticks([])
  plt.yticks([])
  plt.imshow(X[i].reshape(100, 100), cmap="gray")

In [ ]:
# Perform data augmentation with various transformations. In the end, graph 10 as an example
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=15,
    zoom_range=[0.7, 1.4],
    horizontal_flip=True,
    vertical_flip=True,
    rescale=1./255
)

datagen.fit(X)

plt.figure(figsize=(20,8))

for image, label in datagen.flow(X, y, batch_size=10, shuffle=False):
  for i in range(10):
    plt.subplot(2, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.imshow(image[i].reshape(100, 100), cmap="gray")
  break

In [ ]:
denseLayer_AD = tf.keras.models.Sequential([
  tf.keras.layers.Flatten(input_shape=(100, 100, 1)),
  tf.keras.layers.Dense(150, activation='relu'),
  tf.keras.layers.Dense(150, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

CNNmodel_AD = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(100, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

CNN2model_AD = tf.keras.models.Sequential([
  tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(100, 100, 1)),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),
  tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
  tf.keras.layers.MaxPooling2D(2, 2),

  tf.keras.layers.Dropout(0.5),
  tf.keras.layers.Flatten(),
  tf.keras.layers.Dense(250, activation='relu'),
  tf.keras.layers.Dense(1, activation='sigmoid')
])

In [ ]:
denseLayer_AD.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

CNNmodel_AD.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

CNN2model_AD.compile(optimizer='adam',
                    loss='binary_crossentropy',
                    metrics=['accuracy'])

In [ ]:
# First we define the data generator
datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=15,
    zoom_range=[0.7, 1.4],
    horizontal_flip=True,
    vertical_flip=True
)

# Prepare the data
X_train = X[:19700]
X_validation = X[19700:]
y_train = y[:19700]
y_validation = y[19700:]

# Create the iterator
batch_size = 32
data_gen_train = datagen.flow(X_train, y_train, batch_size=batch_size)

# We calculate steps_per_epoch correctly
steps_per_epoch = len(X_train) // batch_size
validation_steps = len(X_validation) // batch_size



## 6. Training with augmentation

In [ ]:
# Dense model training
tensorboardDense_AD = TensorBoard(log_dir='logs/dense_AD')
history_dense = denseLayer_AD.fit(
    data_gen_train,
    epochs=100,
    validation_data=(X_validation, y_validation),
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[tensorboardDense_AD]
)

In [ ]:
# Training the first CNN model
tensorboardCNN_AD = TensorBoard(log_dir='logs/cnn_AD')
history_cnn = CNNmodel_AD.fit(
    data_gen_train,
    epochs=100,
    validation_data=(X_validation, y_validation),
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[tensorboardCNN_AD]
)

In [ ]:
# Training the second CNN model
tensorboardCNN2_AD = TensorBoard(log_dir='logs/cnn2_AD')
history_cnn2 = CNN2model_AD.fit(
    data_gen_train,
    epochs=100,
    validation_data=(X_validation, y_validation),
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=[tensorboardCNN2_AD]
)

## 7. TensorBoard and model export

Convert the saved Keras model to TensorFlow.js for browser inference.

In [ ]:
# Load the colab tensorboard extension
%load_ext tensorboard

In [ ]:
# Run tensorboard and tell it to read the "logs" folder
%tensorboard --logdir logs

In [ ]:
CNNmodel_AD.save('dogs-cats-cnn-ad.h5')

In [ ]:
!mkdir carpeta_salida

In [ ]:
!tensorflowjs_converter --input_format keras dogs-cats-cnn-ad.h5 carpeta_salida